# Tables generator
Generate tables for the paper. This loads files and generates tables that is exported as latex code that can be directly included in the body of the paper.

## Setup

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import yaml
import shutil
import filecmp
from pathlib import Path

In [ ]:
import latexutils.tables as tables

In [ ]:
import pandas as pd

### Folders

In [ ]:
# Check that we are in workspace root by looking for a .tex file
assert any(f.endswith('.tex') for f in os.listdir('.')), "Probably not in workspace root. If this is intended (or you are sure about the directory) you can comment this, otherwise please change to the workspace root directory."

In [ ]:
DATA_DIR = Path('Data')
TABLES_DIR = Path('Tables')

# Check that the directories exist
assert DATA_DIR.is_dir(), f"Data directory '{DATA_DIR}' does not exist. Please create it and add the data files."
assert TABLES_DIR.is_dir(), f"Tables directory '{TABLES_DIR}' does not exist. Please create it to store the generated tables."

## Get data
Fetches data and stores locally. If there are changes to the file: store a backup to catch breakages


Fetches all files in `sources.yaml`

In [ ]:
with open('PythonScripts/sources.yaml') as f:
    sources = yaml.safe_load(f)['sources']

for name, info in sources.items():
    path = info['path']
    filename = info['filename']
    new_path = DATA_DIR / filename
    assert (DATA_DIR / filename).is_file(), f"Source file '{path}' does not exist. Check 'sources.yaml'."

    if (DATA_DIR / filename).is_file():
        if filecmp.cmp(path, new_path, shallow=False):
            print(f"File '{filename}' already exists in '{DATA_DIR}' and is identical. Skipping copy.")
        else:
            print(f"File '{filename}' already exists in '{DATA_DIR}' but differs from source. Making backup and copying.")
            if not (DATA_DIR / "backup_").is_dir(): (DATA_DIR / "backup_").mkdir()
            shutil.copy2(new_path, DATA_DIR / "backup_" / filename, )
            shutil.copy2(path, new_path)
    else:
        print(f"Copying '{filename}' to '{DATA_DIR}'.")
        shutil.copy2(path, new_path)

## Build tables
Meat and bones of the notebook.

This builds the tables that are exported, you need to actually decide what to build here. There are functions available in the `latexutils` package that handle some standard functionality.